<a href="https://colab.research.google.com/github/milleau98/2026-gig-data-analysis/blob/main/notebooks/dataset-generators/google_trends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Collection: Google Trends

This notebook documents how Google trend data was collected using pytrends. The output CSVs are committed to 'data/google_trends/' and do not need to be re-generated to run the analysis notebooks. Re-running this notebook requires a stable internet connection and may be subject to Google rate limiting.



In [1]:
from pytrends.request import TrendReq
import pandas as pd
import time

# configuration
# connect to Google trends (hl = language, tz = timezone offset in minutes)
pytrends = TrendReq(hl='en-US', tz=360)

# anchor keyword used to normalize search interest across groups
# Google trends returns relative scores (0-100) within each batch,
# so we include a common anchor in every group and rescale

anchor = 'Uber'

keyword_groups = [
    [anchor,'Lyft','DoorDash','Instacart'],
    [anchor, 'Fiverr','Upwork','side hustle','gig'],
    [anchor, 'Herbalife','Primerica','Nu skin', 'USANA'],
    [anchor, 'Etsy', 'Shopify', 'Udemy', 'Tupperware']
]

all_data = []

for i, group in enumerate(keyword_groups):
  # timeframe is date range for trend, geo in US to match FRED/yfinance scope
  pytrends.build_payload(group, timeframe='2010-01-01 2025-12-31', geo='US')
  df=pytrends.interest_over_time()
  df=df.drop(columns=['isPartial'])

  # store anchor values as reference scale
  if i == 0:
    anchor_series = df[anchor]
    all_data.append(df)
  else:
    #scale_factor = anchor_series.div(df[anchor]).replace([float("inf"), -float("inf")], 0)
    # subsequent groups: rescale so all keywords are on the same index as the anchor from group 1
    # replace inf (anchor = 0) with NaN to avoid 0ing out low-anchor months
    scale_factor = anchor_series.div(df[anchor]).replace([float("inf"), -float("inf")], float("nan"))
    df = df.multiply(scale_factor, axis=0)
    all_data.append(df.drop(columns=[anchor]))

  # Pause between requests to avoid Google rate limiting
    time.sleep(5)

# combine groups of data into wide df (weekly)
final_data = pd.concat(all_data, axis=1)

final_data.head()

,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy,Tupperware
date,,,,,,,,,,,,,,,,
2010-01-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,13.0,0.0,0.0,3.0
2010-02-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,13.0,0.0,0.0,2.0
2010-03-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,3.0,0.0,1.0,13.0,0.0,0.0,2.0
2010-04-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,12.0,0.0,0.0,2.0
2010-05-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,13.0,0.0,0.0,3.0


In [ ]:
# convert weekly trends to monthly averages
monthly = final_data.resample("ME").mean()

monthly['year'] = monthly.index.year
monthly['month'] = monthly.index.month
monthly['date'] = monthly.index

monthly = monthly.reset_index(drop=True)

monthly.head()

C:\Users\mlxfl\AppData\Local\Temp\ipykernel_2300\1836030185.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = final_data.resample("M").mean()


,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy,Tupperware,year,month,date
0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,13.0,0.0,0.0,3.0,2010,1,2010-01-31
1,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,13.0,0.0,0.0,2.0,2010,2,2010-02-28
2,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,3.0,0.0,1.0,13.0,0.0,0.0,2.0,2010,3,2010-03-31
3,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,12.0,0.0,0.0,2.0,2010,4,2010-04-30
4,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,13.0,0.0,0.0,3.0,2010,5,2010-05-31


In [3]:
# export to csv
import os
from pathlib import Path

# Parent directory
parent_dir = Path.cwd().parent.parent

# create dataset files
monthly.to_csv(os.path.join(parent_dir,'data','google_trends','google_trends_monthly.csv'), index=False)